# Case Study 2 — full pipeline (run top to bottom)

This one notebook runs everything on the university server: build the corpus, embed and index, generate, judge, score, and show the four-bucket result.

**Two switches, set in section 2:**
- **Generator** — the free dev model (Gemini) to shake out bugs now, or the frozen `claude-sonnet-4-6` for the real graded run.
- **Judge** — `stub` (offline, instant) for dev, or the frozen 70B open model via vLLM on this GPU for the real run.

Free-model runs are for **debugging the pipeline, not results**. The numbers that go in the manuscript use the frozen generator + the 70B judge.

Run the cells in order. Sections 3 and 4 build the retrieval store (once). Section 6 generates, section 7 judges and scores, section 8 shows the result.

In [1]:
!pkill -f vllm ; sleep 5 ; nvidia-smi  # kill orpahn server

## 0. Environment probe
Tells us what this server can do. Run it first.

In [2]:
import sys, os, subprocess, platform, urllib.request
print("python:", sys.version.split()[0], "|", platform.platform())
print("cwd:", os.getcwd())
if not os.path.exists("src/run_generation.py"):
    print("!! Run this notebook from the repo ROOT (the folder with src/, config/, test_set.jsonl).")

def check_internet(url="https://pypi.org", timeout=5):
    try:
        urllib.request.urlopen(url, timeout=timeout); return True
    except Exception as e:
        print("  internet check failed:", e); return False

HAS_INTERNET = check_internet()
print("internet:", HAS_INTERNET)

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        p = torch.cuda.get_device_properties(0)
        print("GPU:", torch.cuda.get_device_name(0),
              f"| VRAM {p.total_memory/1e9:.0f} GB | count {torch.cuda.device_count()}")
    else:
        print("GPU: torch present but no CUDA device visible")
except Exception as e:
    print("GPU: torch not importable yet (install deps in section 1) ->", e)
print(f"\nSUMMARY  internet={HAS_INTERNET}  gpu={HAS_GPU}")

python: 3.13.11 | Linux-5.15.0-190-generic-x86_64-with-glibc2.39
cwd: /home/jovyan/case_study2
internet: True
GPU: torch not importable yet (install deps in section 1) -> No module named 'torch'

SUMMARY  internet=True  gpu=False


## 1. Install dependencies (run once, needs internet)
vLLM for the 70B judge is heavy and installed later, only when you switch the judge on.

In [3]:
if HAS_INTERNET:
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
    subprocess.run([sys.executable,"-m","pip","install","-q","openai"], check=False)
    print("core deps installed")
else:
    print("No internet here: install where there is internet, or pre-stage wheels.")

core deps installed


## 2. Config — the only knobs

For the **free dev run** (default): Gemini generator + stub judge. Paste your free Google AI Studio key below.

For the **real graded run**: set `GEN_PROVIDER="anthropic"`, `GEN_MODEL="claude-sonnet-4-6"`, paste `ANTHROPIC_API_KEY`, set `JUDGE="vllm"`, and `RUN_FULL=True`.

In [ ]:
# ---------- GENERATOR ----------
GEN_PROVIDER = "openai_compatible"      # "anthropic" for the frozen graded run
GEN_MODEL    = "openai/gpt-oss-120b"       # "claude-sonnet-4-6" for the frozen run
GEN_BASE_URL = "https://api.groq.com/openai/v1"
GEN_KEY_ENV  = "GROQ_API_KEY"
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")   # <-- paste free key
# os.environ["ANTHROPIC_API_KEY"] = ""   # <-- paste for the frozen run

# ---------- JUDGE ----------
JUDGE          = "vllm"                  # "vllm" for the real 70B judge on this GPU
JUDGE_MODEL    = "Qwen/Qwen2.5-72B-Instruct-AWQ"   # or meta-llama/Llama-3.3-70B-Instruct
JUDGE_BASE_URL = "http://localhost:8000/v1"

# ---------- RUN SIZE ----------
RUN_FULL = True    # False = 8-row sanity per config; True = full 215 x 3

def sh(cmd):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    r = subprocess.run(cmd, shell=not isinstance(cmd, list), capture_output=True, text=True)
    if r.stdout: print(r.stdout[-6000:])
    if r.returncode != 0 and r.stderr: print("STDERR:\n", r.stderr[-4000:])
    return r.returncode

print("generator:", GEN_PROVIDER, GEN_MODEL, "| judge:", JUDGE, "| full run:", RUN_FULL)

generator: openai_compatible openai/gpt-oss-120b | judge: vllm | full run: True


## 3. Corpus (400 chunks from the frozen guidelines)
Uses the committed corpus if present; only rebuilds if missing (rebuild needs poppler/pdftotext).

In [5]:
CHUNKS = "results/corpus_chunks.jsonl"
PDF = "data/guidelines/Draft_Guidelines_on_the_classification_of_high_risk_AI_Annex_III.pdf"
if os.path.exists(CHUNKS):
    print("corpus present (committed):", sum(1 for _ in open(CHUNKS)), "chunks — skip rebuild")
else:
    sh([sys.executable, "src/build_corpus.py", PDF, CHUNKS])
    print("chunks:", sum(1 for _ in open(CHUNKS)))

corpus present (committed): 400 chunks — skip rebuild


## 4. Embed + index (downloads bge model, needs internet, builds Chroma)
The vector store is not in git, so build it here once.

In [6]:
sh([sys.executable, "src/embed_and_index.py", "config/pipeline.yaml"])

$ /opt/conda/bin/python src/embed_and_index.py config/pipeline.yaml
loaded 400 chunks
embedded -> (400, 768)
persisted 400 vectors -> results/chroma/eu_ai_act_guidelines



0

## 5. Retrieval quality check (optional, no API needed)
Should show Hit@5 around 0.83.

In [7]:
sh([sys.executable, "src/retrieval_eval.py", "config/pipeline.yaml"])

$ /opt/conda/bin/python src/retrieval_eval.py config/pipeline.yaml

Retrieval quality over 215 rows
  Hit@5   0.833   (PRIMARY)
  Hit@10  0.898
  Recall@5  0.246   Recall@10 0.374
  MRR     0.633
  wrote results/retrieval_eval.json and results/retrieval_eval.md



0

In [8]:
# patch: ride out the free-tier 5-requests-per-minute limit
import pathlib
p = pathlib.Path("src/run_generation.py")
s = p.read_text()
s = s.replace("def call_generator(client, kind, gen_cfg, system, user, max_retries=4):",
              "def call_generator(client, kind, gen_cfg, system, user, max_retries=8):")
s = s.replace("            time.sleep(2 ** attempt)",
              "            time.sleep(15)")
p.write_text(s)
print("patched: 8 retries, 15s wait on rate-limit")

patched: 8 retries, 15s wait on rate-limit


## 6. Generation
Writes one file per config to results/runs/. Sanity (8 rows) unless RUN_FULL=True.

In [16]:
gen_args = [sys.executable, "src/run_generation.py",
            "--provider", GEN_PROVIDER, "--model", GEN_MODEL,
            "--base-url", GEN_BASE_URL, "--api-key-env", GEN_KEY_ENV]
if not RUN_FULL:
    gen_args += ["--limit", "8"]
sh(gen_args)

import glob, json
for f in sorted(glob.glob("results/runs/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), "->", len(rows), "rows | first: pred=%s gold=%s" %
          (rows[0]["pred_label"], rows[0]["gold_label"]))

$ /opt/conda/bin/python src/run_generation.py --provider openai_compatible --model openai/gpt-oss-120b --base-url https://api.groq.com/openai/v1 --api-key-env GROQ_API_KEY
ed=not-high-risk gold=not-high-risk ok
  [163/215] anx3-165: pred=high-risk gold=not-high-risk ok
  [164/215] anx3-166: pred=high-risk gold=not-high-risk ok
  [165/215] anx3-167: pred=high-risk gold=not-high-risk ok
  [166/215] anx3-168: pred=high-risk gold=not-high-risk ok
  [167/215] anx3-169: pred=high-risk gold=not-high-risk ok
  [168/215] anx3-170: pred=high-risk gold=not-high-risk ok
  [169/215] anx3-171: pred=not-high-risk gold=not-high-risk ok
  [170/215] anx3-172: pred=high-risk gold=not-high-risk ok
  [171/215] anx3-173: pred=high-risk gold=not-high-risk ok
  [172/215] anx3-174: pred=not-high-risk gold=not-high-risk ok
  [173/215] anx3-175: pred=not-high-risk gold=not-high-risk ok
  [174/215] anx3-176: pred=high-risk gold=not-high-risk ok
  [175/215] anx3-177: pred=high-risk gold=not-high-risk ok
  [176/215

## 7. Judge + scoring
`stub` = offline and instant (dev). `vllm` = the real 70B judge on this GPU: the next cell starts a vLLM server (first run downloads the 70B, can take a while), scores against it, then stops it.

In [21]:
import time, urllib.request, os
vllm_env = {**os.environ, "VLLM_USE_FLASHINFER_SAMPLER": "0"}
vllm_proc = None
if JUDGE == "vllm":
    if not HAS_GPU:
        print("JUDGE=vllm but no GPU detected.")
    else:
        subprocess.run([sys.executable,"-m","pip","install","-q","vllm"], check=False)
        vllm_proc = subprocess.Popen([sys.executable,"-m","vllm.entrypoints.openai.api_server",
            "--model", JUDGE_MODEL, "--port", "8000",
            "--dtype", "auto",
            "--gpu-memory-utilization", "0.95",
            "--max-model-len", "6144"],
            env=vllm_env)                      # <-- disables the FlashInfer sampler JIT
        print("starting vLLM (single GPU, no flashinfer sampler)...")
        up = False
        for _ in range(300):
            try:
                urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3)
                up = True; print("vLLM server is up"); break
            except Exception:
                time.sleep(10)
        if not up:
            print("vLLM did NOT come up — scroll up for the real error.")
else:
    print("JUDGE=stub — scoring runs offline and instant.")

JUDGE=vllm but no GPU detected.


In [18]:
score_args = [sys.executable, "src/run_scoring.py", "--judge", JUDGE]
if JUDGE == "vllm":
    score_args += ["--judge-model", JUDGE_MODEL, "--judge-base-url", JUDGE_BASE_URL]
sh(score_args)

if vllm_proc is not None:
    vllm_proc.terminate(); print("vLLM server stopped")

$ /opt/conda/bin/python src/run_scoring.py --judge vllm --judge-model Qwen/Qwen2.5-72B-Instruct-AWQ --judge-base-url http://localhost:8000/v1
scoring runs in results/runs  ->  results/scoring
judge: vllm (Qwen/Qwen2.5-72B-Instruct-AWQ) @ http://localhost:8000/v1

[1/3] correctness
    baseline1_plain_llm      acc=0.712  HR_f1=0.670  parse_fail=0
    baseline2_standard_rag   acc=0.911  HR_f1=0.905  parse_fail=0

[2/3] faithfulness
  faithfulness: baseline2_standard_rag (45 rows) judge=openai-compatible:Qwen/Qwen2.5-72B-Instruct-AWQ

STDERR:
 
        follow_redirects=follow_redirects,
        history=history,
    )
  File "/opt/conda/lib/python3.13/site-packages/httpx2/_client.py", line 1043, in _send_handling_redirects
    response = self._send_single_request(request)
  File "/opt/conda/lib/python3.13/site-packages/httpx2/_client.py", line 1076, in _send_single_request
    response = transport.handle_request(request)
  File "/opt/conda/lib/python3.13/site-packages/httpx2/_transports/de

## 8. Results — the four-bucket matrix
Correctness, faithfulness, then the right/wrong x faithful/unfaithful buckets. The off-diagonal rows are in results/scoring/buckets/*_offdiagonal.jsonl.

In [19]:
for f in ["results/scoring/correctness_summary.md",
          "results/scoring/faithfulness_summary.md",
          "results/scoring/buckets_summary.md"]:
    print("="*72); print(f); print("="*72)
    print(open(f).read() if os.path.exists(f) else "(not produced yet)")
    print()

results/scoring/correctness_summary.md
# Correctness (predicted label vs frozen Commission ground truth)

Positive class = high-risk. Accuracy plus per-class precision/recall/F1 because the set is imbalanced (frozen). Parse failures counted as incorrect and also shown separately.

| Config | n | Accuracy | HR precision | HR recall | HR F1 | Macro F1 | Parse fails |
|---|---|---|---|---|---|---|---|
| baseline1_plain_llm | 215 | 0.712 | 0.516 | 0.955 | 0.670 | 0.707 | 0 |
| baseline2_standard_rag | 45 | 0.911 | 0.950 | 0.864 | 0.905 | 0.911 | 0 |

## baseline1_plain_llm

Confusion (high-risk positive): tp=63 fp=59 fn=3 tn=90

By edge-case type: article-6-3-filter n=41 acc=0.854, none n=174 acc=0.678

By area: biometrics 0.438, critical-infrastructure 0.500, education 0.893, employment 0.969, essential-services 0.837, justice-democracy 0.688, law-enforcement 0.348, migration 0.826

## baseline2_standard_rag

Confusion (high-risk positive): tp=19 fp=1 fn=3 tn=22

By edge-case type: none n

In [35]:
nvidia-smi

NameError: name 'nvidia' is not defined

In [ ]:
curl -s http://localhost:8000/v1/models

In [22]:
# in a cell
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [23]:
# re-run section 0, or just:
import torch; print(torch.__version__, torch.cuda.is_available())

2.14.0+cu130 False
